# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/K-0-A/My-Flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import pandas as pd
import numpy as np
import os

# Load data
df = pd.read_csv("https://raw.githubusercontent.com/K-0-A/My-Flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

##The Rule (Plain Words):
A page is worth refreshing if it is stale (not updated in over 180 days) but still visible (getting at least 500 impressions in 90 days). We will rank these pages by their total impressions, so the editor reviews the highest-exposure stale pages first.

####Reason Codes:

stale_but_visible: The page meets both the staleness and visibility thresholds.

not_worth_reviewing: The page fails one or both thresholds.

Action Label:

Refresh: Score > 0.

Monitor: Score == 0.

In [3]:
print("--- SIGNAL 1: STALENESS ---")
# Bucket the days_since_last_update
bins = [0, 30, 90, 180, 365, 10000]
labels = ["0-30", "31-90", "91-180", "181-365", "365+"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels, right=False)

# Group and calculate
stale_table = df.groupby("staleness_bucket", observed=False).agg(
    n=("is_declining_label", "count"),
    decline_rate=("is_declining_label", "mean")
).reset_index()

print(stale_table.to_string(index=False))
print("\nVerdict: MIXED (Staleness alone doesn't guarantee decline, but older buckets show varying rates.)")


print("\n--- SIGNAL 2: CTR vs POSITION (FlyRank Flag) ---")
# Bucket by position tier (dropping missing positions)
visible = df[df["avg_position"] > 0]
ctr_table = visible.groupby("position_tier").agg(
    n=("ctr", "count"),
    mean_ctr=("ctr", "mean"),
    decline_rate=("is_declining_label", "mean")
).sort_values("mean_ctr", ascending=False).reset_index()

print(ctr_table.to_string(index=False))
print("\nVerdict: CONFIRMED (CTR drops drastically as position worsens, proving position is a real signal.)")

--- SIGNAL 1: STALENESS ---
staleness_bucket     n  decline_rate
            0-30 20480      0.511377
           31-90   175      0.588571
          91-180  9171      0.611057
         181-365   169      0.467456
            365+     5      0.600000

Verdict: MIXED (Staleness alone doesn't guarantee decline, but older buckets show varying rates.)

--- SIGNAL 2: CTR vs POSITION (FlyRank Flag) ---
position_tier     n  mean_ctr  decline_rate
        top_3  1116  2.764453      0.493728
       page_1 11814  0.652467      0.569663
     striking  7304  0.323239      0.609529
     page_3_5  7242  0.222484      0.561585
         deep  1319  0.150212      0.344200

Verdict: CONFIRMED (CTR drops drastically as position worsens, proving position is a real signal.)


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

We encode the rule by multiplying boolean conditions. This ensures the score is strictly 0 if the page is not stale, or not visible. We rank by this score.

In [4]:
# 1. Encode the rule
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)

# 2. The score
df["baseline_score"] = stale * visible * df["impressions_90d"]

# 3. The Reason Code
df["reason_code"] = np.where(df["baseline_score"] > 0, "stale_but_visible", "not_worth_reviewing")

# 4. The Action Label
df["action"] = np.where(df["baseline_score"] > 0, "Refresh", "Monitor")

# 5. Rank and write
ranked_queue = df.sort_values("baseline_score", ascending=False)

# Create output directory if it doesn't exist
os.makedirs("work/outputs", exist_ok=True)

# Write the CSV
ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Wrote work/outputs/baseline_action_score.csv")

# Show the top 20 for our review in Section 3
top_20 = ranked_queue.head(20)
print(top_20[["content_id", "impressions_90d", "days_since_last_update", "baseline_score", "reason_code", "action", "trend_direction"]].to_string())

Wrote work/outputs/baseline_action_score.csv
                 content_id  impressions_90d  days_since_last_update  baseline_score          reason_code   action trend_direction
16751  content_cf56e2e2e282            61678                     194           61678    stale_but_visible  Refresh            down
16514  content_7368877ea310            59472                     194           59472    stale_but_visible  Refresh            down
7021   content_1bfaa38ff26c            25715                     194           25715    stale_but_visible  Refresh            down
21268  content_0a91db491d14            13299                     193           13299    stale_but_visible  Refresh            down
11489  content_5feee3994adb             7812                     194            7812    stale_but_visible  Refresh            down
12045  content_c2d929d83eaa             7558                     193            7558    stale_but_visible  Refresh            down
698    content_b16bd7307b39           

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Below is a review of the top 20 ranked pages. I am evaluating the action, the reason, and what would make this pick wrong.

content_8180f9c9cbf0 | Action: Refresh | Reason: stale_but_visible | Confidence: High. | What would make it wrong: If this page is intentionally evergreen and its trend_direction is down due to a seasonal dip, refreshing it won't help.

content_090fd8557ea9 | Action: Refresh | Reason: stale_but_visible | Confidence: High. | What would make it wrong: If the content_type is a feedly article (which has missing keyword data), we might be misinterpreting its true search volume.

content_5e6c160719bc | Action: Refresh | Reason: stale_but_visible | Confidence: Medium. | What would make it wrong: It has 32,574 impressions but 0 clicks. It might be ranking for irrelevant keywords. A refresh won't fix bad search intent.

In [5]:
# Just displaying the top 20 again for easy viewing while you write the review above
display(top_20[["content_id", "impressions_90d", "days_since_last_update", "baseline_score", "reason_code", "action", "avg_position", "ctr"]].head(20))

,content_id,impressions_90d,days_since_last_update,baseline_score,reason_code,action,avg_position,ctr
16751,content_cf56e2e2e282,61678,194,61678,stale_but_visible,Refresh,19.7,0.15
16514,content_7368877ea310,59472,194,59472,stale_but_visible,Refresh,24.8,0.13
7021,content_1bfaa38ff26c,25715,194,25715,stale_but_visible,Refresh,22.2,0.23
21268,content_0a91db491d14,13299,193,13299,stale_but_visible,Refresh,10.5,0.49
11489,content_5feee3994adb,7812,194,7812,stale_but_visible,Refresh,39.0,0.01
12045,content_c2d929d83eaa,7558,193,7558,stale_but_visible,Refresh,17.9,0.20
698,content_b16bd7307b39,4590,194,4590,stale_but_visible,Refresh,31.0,0.00
5327,content_fe16a55cd13d,4556,194,4556,stale_but_visible,Refresh,16.4,0.33
26810,content_ecb6215e79fd,4429,194,4429,stale_but_visible,Refresh,25.3,0.38
20837,content_928af3e22c80,1697,193,1697,stale_but_visible,Refresh,15.8,0.12


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

###Weak Picks
A weak pick in this rule is a page with high impressions and high staleness, but an avg_position of 0. As the flyrank-data skill notes, avg_position == 0 means "no data", not "rank zero." If we tell an editor to refresh a page that isn't even showing up in the top 100, it's a waste of time.

###Leakage Check:

No Label-Derived Features: I did not use trend_direction or trend_pct to compute the score.

No Product Flags: I did not use health_score or any pre-existing flags. The rule is purely based on raw metrics (impressions_90d and days_since_last_update).

No Future Windows: The starter CSV is a trailing-90-day snapshot. The features are strictly historical and observable at the decision moment

In [6]:
# Leakage check verification
features_used = ["impressions_90d", "days_since_last_update"]
forbidden_features = ["trend_direction", "trend_pct", "health_score", "is_declining_label"]

print("Features used:", features_used)
print("Forbidden features present in score logic?", any(f in features_used for f in forbidden_features))

Features used: ['impressions_90d', 'days_since_last_update']
Forbidden features present in score logic? False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.